In [1]:
import wandb
import pandas as pd
import numpy as np

# 1. Setup
api = wandb.Api()
entity = "luis-perdigao-instituto-superior-t-cnico"
project = "PCS_ET_v3"  # <--- Updated to v3

print(f"Connecting to {entity}/{project}...")
try:
    runs = api.runs(f"{entity}/{project}")
    print(f"Found {len(runs)} runs. Retrieving metrics...")
except Exception as e:
    print(f"Error: {e}")
    exit()

data_list = []

for i, run in enumerate(runs):
    if i % 5 == 0: print(f"Processing run {i}/{len(runs)}...")

    # A. Get Config
    config = {k: v for k, v in run.config.items() if not k.startswith('_')}
    
    # B. Fetch History (Validation selection logic)
    keys = ["accuracy_validation", "accuracy_test", "c_accuracy_test"]
    history = pd.DataFrame(run.scan_history(keys=keys))
    
    best_val_acc = np.nan
    final_rank_acc = np.nan
    final_class_acc = np.nan
    
    if not history.empty and "accuracy_validation" in history.columns:
        # 1. Find epoch with max Validation Accuracy
        best_epoch_idx = history["accuracy_validation"].idxmax()
        
        # 2. Extract Test scores from that SAME epoch
        if "accuracy_test" in history.columns:
            final_rank_acc = history.loc[best_epoch_idx, "accuracy_test"]
        
        if "c_accuracy_test" in history.columns:
            final_class_acc = history.loc[best_epoch_idx, "c_accuracy_test"]
            
    else:
        # Fallback to summary if history is empty
        final_rank_acc = run.summary.get("accuracy_test", np.nan)
        final_class_acc = run.summary.get("c_accuracy_test", np.nan)

    # C. Build Row
    data_list.append({
        "run_name": run.name,
        "backbone": config.get("backbone", "unknown"),
        "pooling": config.get("pooling", "unknown"),
        "augment": config.get("augment", "none"),   # <--- Capture Augmentation
        "num_ft_blocks": config.get("num_ft_blocks", 0),
        "test_rank_acc": final_rank_acc,
        "test_class_acc": final_class_acc
    })

# 3. Save
df = pd.DataFrame(data_list)
df.to_csv("augmentation_results.csv", index=False)
print("Done! Saved 'augmentation_results.csv'.")

wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: luis-perdigao (luis-perdigao-instituto-superior-t-cnico) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Connecting to luis-perdigao-instituto-superior-t-cnico/PCS_ET_v3...
Found 6 runs. Retrieving metrics...
Processing run 0/6...
Processing run 5/6...
Done! Saved 'augmentation_results.csv'.


In [3]:
import pandas as pd
import numpy as np

def generate_aug_table(csv_path="augmentation_results.csv"):
    df = pd.read_csv(csv_path)

    # 1. Clean Names
    pooling_map = {'cls': 'CLS', 'concat': 'Concat', 'topk': 'Top-K'}
    aug_map = {'light': 'Light', 'heavy': 'Heavy', 'none': 'None'}
    
    df['pooling'] = df['pooling'].map(pooling_map).fillna(df['pooling'])
    df['augment'] = df['augment'].map(aug_map).fillna(df['augment'])

    # 2. Pivot Tables
    # Rows: Pooling | Columns: Augmentation
    pivot_rank = df.pivot_table(index='pooling', columns='augment', values='test_rank_acc') * 100
    pivot_class = df.pivot_table(index='pooling', columns='augment', values='test_class_acc') * 100

    # 3. Generate LaTeX
    latex_str = []
    latex_str.append("\\begin{table}[ht]")
    latex_str.append("\\centering")
    latex_str.append("\\caption{Effect of Data Augmentation on DINOv3 (4 Blocks Unfrozen). Values are Rank / Class Accuracy (\\%).}")
    latex_str.append("\\label{tab:augmentation_results}")
    
    # Dynamic column setup based on what's in the CSV
    cols = sorted(pivot_rank.columns.tolist(), reverse=True) # e.g., Light, Heavy
    col_def = "l" + "c" * len(cols)
    
    latex_str.append(f"\\begin{{tabular}}{{{col_def}}}")
    latex_str.append("\\toprule")
    latex_str.append(f"Pooling & {' & '.join(cols)} \\\\")
    latex_str.append("\\midrule")
    
    for pooling in pivot_rank.index:
        row_values = []
        for col in cols:
            val_r = pivot_rank.loc[pooling, col]
            val_c = pivot_class.loc[pooling, col]
            
            if np.isnan(val_r) or np.isnan(val_c):
                row_values.append("-")
            else:
                # Highlight if it's the best across the row (Comparison of Augmentations)
                # Note: You might prefer column-wise highlighting; let me know.
                is_max_r = (val_r == pivot_rank.loc[pooling].max())
                is_max_c = (val_c == pivot_class.loc[pooling].max())
                
                s_r = f"{val_r:.2f}"
                s_c = f"{val_c:.2f}"
                
                if is_max_r: s_r = f"\\textbf{{{s_r}}}"
                if is_max_c: s_c = f"\\textbf{{{s_c}}}"
                
                row_values.append(f"{s_r} / {s_c}")
        
        latex_str.append(f"{pooling} & {' & '.join(row_values)} \\\\")

    latex_str.append("\\bottomrule")
    latex_str.append("\\end{tabular}")
    latex_str.append("\\end{table}")

    return "\n".join(latex_str)

print(generate_aug_table())

\begin{table}[ht]
\centering
\caption{Effect of Data Augmentation on DINOv3 (4 Blocks Unfrozen). Values are Rank / Class Accuracy (\%).}
\label{tab:augmentation_results}
\begin{tabular}{lcc}
\toprule
Pooling & Light & Heavy \\
\midrule
CLS & 76.54 / \textbf{76.80} & \textbf{76.71} / 75.51 \\
Concat & \textbf{76.28} / \textbf{76.97} & 75.68 / 73.12 \\
Top-K & 75.51 / \textbf{76.54} & \textbf{75.68} / 76.28 \\
\bottomrule
\end{tabular}
\end{table}
